# 🧠 Customer Feedback Analyzer
**NLP + LLM Pipeline | Google Colab**

**Features:**
- ✅ Sentiment Analysis (multilingual, incl. Amharic)
- ✅ Topic / Category Classification
- ✅ Key Phrase Extraction
- ✅ Auto Summarization & Insights
- ✅ Amharic Language Support
- ✅ REST API Server (served to Spring Boot backend)

**Supported Feedback Types:** Product reviews, Support tickets, Survey responses, Social media comments


## 1. Install Dependencies

In [1]:
!pip install -q "transformers==4.44.2" \
               "sentence-transformers==3.0.1" \
               torch sentencepiece flask flask-cors pyngrok \
               langdetect keybert \
               openai anthropic python-dotenv \
               pandas numpy scikit-learn

print('✅ Done! Now go to Runtime → Restart session, then continue from Cell 2.')

✅ Done! Now go to Runtime → Restart session, then continue from Cell 2.


## 2. Load Models

In [1]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM
)
from langdetect import detect
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

device = 0 if torch.cuda.is_available() else -1
print(f'🔧 Device: {"GPU" if device == 0 else "CPU"}')

print('📥 Loading multilingual sentiment model...')
# XLM-RoBERTa: multilingual, supports 100+ languages including Amharic
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
    device=device,
    truncation=True,
    max_length=512
)

print('📥 Loading summarization model...')
summarizer = pipeline(
    'summarization',
    model='facebook/bart-large-cnn',
    device=device,
    truncation=True
)

print('📥 Loading zero-shot classification model...')
classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=device
)

print('📥 Loading KeyBERT for key phrase extraction...')
kw_model = KeyBERT(model='paraphrase-multilingual-MiniLM-L12-v2')

print('✅ All models loaded!')

🔧 Device: GPU
📥 Loading multilingual sentiment model...
📥 Loading summarization model...
📥 Loading zero-shot classification model...
📥 Loading KeyBERT for key phrase extraction...
✅ All models loaded!


## 3. Amharic Language Support Setup

In [2]:
import subprocess
subprocess.run(['pip', 'install', 'deep-translator', 'langid', '-q'], check=True)

from deep_translator import GoogleTranslator
import langid

print('📥 Initializing Amharic → English translation...')

def translate_amharic_to_english(text: str) -> str:
    """Translate Amharic text to English for NLP processing."""
    try:
        result = GoogleTranslator(source='am', target='en').translate(text)
        return result
    except Exception as e:
        print(f'⚠️ Amharic translation error: {e}')
        return text

def detect_language(text: str) -> str:
    """Detect language of input text."""
    try:
        lang, _ = langid.classify(text)
        return lang
    except:
        return 'unknown'

print('✅ Amharic support ready!')

# Test Amharic detection
sample_amharic = 'ይህ ምርት በጣም ጥሩ ነው'
print(f'Sample Amharic: {sample_amharic}')
print(f'Detected language: {detect_language(sample_amharic)}')  # expects: am
translated = translate_amharic_to_english(sample_amharic)
print(f'Translated: {translated}')

📥 Initializing Amharic → English translation...
✅ Amharic support ready!
Sample Amharic: ይህ ምርት በጣም ጥሩ ነው
Detected language: am
Translated: This product is very good


## 4. Core NLP Analysis Functions

In [3]:
from typing import Dict, List, Any
import re

# ─── Topic categories per feedback type ────────────────────────────────────
TOPIC_LABELS = {
    'product_review':  ['quality', 'price', 'delivery', 'packaging', 'functionality',
                        'customer service', 'durability', 'design', 'value for money'],
    'support_ticket':  ['billing issue', 'technical problem', 'account access',
                        'feature request', 'bug report', 'refund request',
                        'shipping issue', 'cancellation'],
    'survey_response': ['user experience', 'product satisfaction', 'support quality',
                        'pricing feedback', 'feature suggestions', 'overall satisfaction'],
    'social_media':    ['brand reputation', 'product complaint', 'praise',
                        'comparison', 'question', 'misinformation', 'viral content'],
    'general':         ['positive experience', 'negative experience', 'neutral feedback',
                        'suggestion', 'complaint', 'compliment', 'question']
}

SENTIMENT_EMOJI = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}

# ─── Sentiment Analysis ────────────────────────────────────────────────────
def analyze_sentiment(text: str, lang: str = None) -> Dict:
    """Perform multilingual sentiment analysis."""
    if lang is None:
        lang = detect_language(text)

    processing_text = text
    translated_from_amharic = False

    if lang == 'am':  # Amharic
        processing_text = translate_amharic_to_english(text)
        translated_from_amharic = True

    result = sentiment_pipeline(processing_text[:512])[0]
    label = result['label'].lower()

    # Normalize labels (model may use positive/negative/neutral)
    if 'pos' in label:  label = 'positive'
    elif 'neg' in label: label = 'negative'
    else: label = 'neutral'

    return {
        'sentiment': label,
        'confidence': round(result['score'], 4),
        'emoji': SENTIMENT_EMOJI[label],
        'language': lang,
        'amharic_translated': translated_from_amharic
    }

# ─── Topic Classification ──────────────────────────────────────────────────
def classify_topic(text: str, feedback_type: str = 'general', lang: str = None) -> Dict:
    """Zero-shot topic classification."""
    if lang is None:
        lang = detect_language(text)

    processing_text = text
    if lang == 'am':
        processing_text = translate_amharic_to_english(text)

    labels = TOPIC_LABELS.get(feedback_type, TOPIC_LABELS['general'])
    result = classifier(processing_text[:512], candidate_labels=labels)

    top3 = [
        {'topic': lbl, 'score': round(score, 4)}
        for lbl, score in zip(result['labels'][:3], result['scores'][:3])
    ]
    return {
        'primary_topic': result['labels'][0],
        'confidence': round(result['scores'][0], 4),
        'top_topics': top3,
        'feedback_type': feedback_type
    }

# ─── Key Phrase Extraction ─────────────────────────────────────────────────
def extract_keyphrases(text: str, lang: str = None, top_n: int = 8) -> Dict:
    """Extract key phrases using KeyBERT."""
    if lang is None:
        lang = detect_language(text)

    processing_text = text
    if lang == 'am':
        processing_text = translate_amharic_to_english(text)

    if len(processing_text.split()) < 3:
        return {'keyphrases': [], 'note': 'Text too short for extraction'}

    keywords = kw_model.extract_keywords(
        processing_text,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        use_mmr=True,
        diversity=0.6,
        top_n=top_n
    )
    return {
        'keyphrases': [{'phrase': kw, 'relevance': round(score, 4)} for kw, score in keywords]
    }

# ─── Auto Summarization ───────────────────────────────────────────────────
def summarize_feedback(text: str, lang: str = None) -> Dict:
    """Summarize feedback text."""
    if lang is None:
        lang = detect_language(text)

    processing_text = text
    if lang == 'am':
        processing_text = translate_amharic_to_english(text)

    word_count = len(processing_text.split())
    if word_count < 30:
        return {'summary': processing_text, 'note': 'Text too short to summarize'}

    max_len = min(130, max(30, word_count // 3))
    min_len = min(25, max_len - 5)

    result = summarizer(processing_text[:1024], max_length=max_len, min_length=min_len, do_sample=False)
    return {'summary': result[0]['summary_text']}

print('✅ Analysis functions defined!')

✅ Analysis functions defined!


## 5. Full Analysis Pipeline

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# COMPLETE FEEDBACK ANALYSIS PIPELINE — Single Cell
# ═══════════════════════════════════════════════════════════════════════════

import subprocess
subprocess.run(['pip', 'install', 'deep-translator', 'langid', 'keybert', '-q'], check=True)

import time
import torch
import langid
from typing import Dict, List
from deep_translator import GoogleTranslator
from transformers import pipeline as hf_pipeline
from keybert import KeyBERT
from collections import Counter

# ── Device ────────────────────────────────────────────────────────────────
device = 0 if torch.cuda.is_available() else -1
print(f'🖥️  Device: {"GPU" if device == 0 else "CPU"}')

# ── Load Models ───────────────────────────────────────────────────────────
print('📥 Loading sentiment model...')
sentiment_pipeline = hf_pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=device
)

print('📥 Loading zero-shot classifier...')
classifier = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=device
)

print('📥 Loading summarizer...')
summarizer = hf_pipeline(
    'summarization',
    model='facebook/bart-large-cnn',
    device=device
)

print('📥 Loading KeyBERT...')
kw_model = KeyBERT()

print('✅ All models loaded!')

# ── Topic Labels ──────────────────────────────────────────────────────────
TOPIC_LABELS = {
    'product_review':  ['quality', 'price', 'delivery', 'packaging', 'functionality',
                        'customer service', 'durability', 'design', 'value for money'],
    'support_ticket':  ['billing issue', 'technical problem', 'account access',
                        'feature request', 'bug report', 'refund request',
                        'shipping issue', 'cancellation'],
    'survey_response': ['user experience', 'product satisfaction', 'support quality',
                        'pricing feedback', 'feature suggestions', 'overall satisfaction'],
    'social_media':    ['brand reputation', 'product complaint', 'praise',
                        'comparison', 'question', 'misinformation', 'viral content'],
    'general':         ['positive experience', 'negative experience', 'neutral feedback',
                        'suggestion', 'complaint', 'compliment', 'question']
}

SENTIMENT_EMOJI = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}

# ── Language Helpers ──────────────────────────────────────────────────────
def detect_language(text: str) -> str:
    try:
        lang, _ = langid.classify(text)
        return lang
    except:
        return 'unknown'

def translate_amharic_to_english(text: str) -> str:
    try:
        return GoogleTranslator(source='am', target='en').translate(text)
    except Exception as e:
        print(f'⚠️ Amharic translation error: {e}')
        return text

# ── Analysis Functions ────────────────────────────────────────────────────
def analyze_sentiment(text: str, lang: str = None) -> Dict:
    if lang is None:
        lang = detect_language(text)
    processing_text = text
    translated_from_amharic = False
    if lang == 'am':
        processing_text = translate_amharic_to_english(text)
        translated_from_amharic = True
    result = sentiment_pipeline(processing_text[:512])[0]
    label  = result['label'].lower()
    if 'pos' in label:   label = 'positive'
    elif 'neg' in label: label = 'negative'
    else:                label = 'neutral'
    return {
        'sentiment':          label,
        'confidence':         round(result['score'], 4),
        'emoji':              SENTIMENT_EMOJI[label],
        'language':           lang,
        'amharic_translated': translated_from_amharic
    }

def classify_topic(text: str, feedback_type: str = 'general', lang: str = None) -> Dict:
    if lang is None:
        lang = detect_language(text)
    processing_text = text
    if lang == 'am':
        processing_text = translate_amharic_to_english(text)
    labels = TOPIC_LABELS.get(feedback_type, TOPIC_LABELS['general'])
    result = classifier(processing_text[:512], candidate_labels=labels)
    top3 = [
        {'topic': lbl, 'score': round(score, 4)}
        for lbl, score in zip(result['labels'][:3], result['scores'][:3])
    ]
    return {
        'primary_topic': result['labels'][0],
        'confidence':    round(result['scores'][0], 4),
        'top_topics':    top3,
        'feedback_type': feedback_type
    }

def extract_keyphrases(text: str, lang: str = None, top_n: int = 8) -> Dict:
    if lang is None:
        lang = detect_language(text)
    processing_text = text
    if lang == 'am':
        processing_text = translate_amharic_to_english(text)
    if len(processing_text.split()) < 3:
        return {'keyphrases': [], 'note': 'Text too short for extraction'}
    keywords = kw_model.extract_keywords(
        processing_text,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        use_mmr=True,
        diversity=0.6,
        top_n=top_n
    )
    return {
        'keyphrases': [{'phrase': kw, 'relevance': round(score, 4)} for kw, score in keywords]
    }

def summarize_feedback(text: str, lang: str = None) -> Dict:
    if lang is None:
        lang = detect_language(text)
    processing_text = text
    if lang == 'am':
        processing_text = translate_amharic_to_english(text)
    word_count = len(processing_text.split())
    if word_count < 30:
        return {'summary': processing_text, 'note': 'Text too short to summarize'}
    max_len = min(130, max(30, word_count // 3))
    min_len = min(25, max_len - 5)
    result = summarizer(processing_text[:1024], max_length=max_len, min_length=min_len, do_sample=False)
    return {'summary': result[0]['summary_text']}

# ── Pipeline ──────────────────────────────────────────────────────────────
def analyze_feedback(text: str, feedback_type: str = 'general', source: str = None) -> Dict:
    start = time.time()
    text  = text.strip()
    if not text:
        return {'error': 'Empty feedback text'}
    lang       = detect_language(text)
    sentiment  = analyze_sentiment(text, lang)
    topic      = classify_topic(text, feedback_type, lang)
    keyphrases = extract_keyphrases(text, lang)
    summary    = summarize_feedback(text, lang)
    elapsed    = round(time.time() - start, 2)
    return {
        'input': {
            'text':          text,
            'feedback_type': feedback_type,
            'source':        source or 'unknown',
            'language':      lang,
            'word_count':    len(text.split())
        },
        'sentiment':  sentiment,
        'topic':      topic,
        'keyphrases': keyphrases,
        'summary':    summary,
        'meta': {'processing_time_sec': elapsed, 'model_device': 'GPU' if device == 0 else 'CPU'}
    }

def batch_analyze(feedbacks: List[Dict]) -> List[Dict]:
    results = []
    for i, item in enumerate(feedbacks):
        print(f'  Processing {i+1}/{len(feedbacks)}...')
        results.append(analyze_feedback(
            text=item.get('text', ''),
            feedback_type=item.get('feedback_type', 'general'),
            source=item.get('source', 'unknown')
        ))
    return results

def generate_insights(results: List[Dict]) -> Dict:
    if not results:
        return {}
    sentiments  = [r['sentiment']['sentiment'] for r in results if 'sentiment' in r]
    topics      = [r['topic']['primary_topic']  for r in results if 'topic' in r]
    all_phrases = [
        kp['phrase']
        for r in results if 'keyphrases' in r
        for kp in r['keyphrases'].get('keyphrases', [])
    ]
    sentiment_counts = dict(Counter(sentiments))
    topic_counts     = dict(Counter(topics).most_common(5))
    top_phrases      = dict(Counter(all_phrases).most_common(10))
    total   = len(sentiments)
    pos_pct = round(sentiment_counts.get('positive', 0) / total * 100, 1) if total else 0
    neg_pct = round(sentiment_counts.get('negative', 0) / total * 100, 1) if total else 0
    return {
        'total_feedbacks':        total,
        'sentiment_distribution': sentiment_counts,
        'positive_rate_pct':      pos_pct,
        'negative_rate_pct':      neg_pct,
        'top_topics':             topic_counts,
        'top_keyphrases':         top_phrases,
        'alert': 'High negative feedback detected!' if neg_pct > 40 else None
    }

print('✅ Full pipeline ready!')

# ── Quick Demo ────────────────────────────────────────────────────────────
print('\n--- Demo: English Product Review ---')
demo_en = analyze_feedback(
    text='This product is absolutely amazing! Great quality, fast delivery, and excellent customer support. Highly recommend!',
    feedback_type='product_review',
    source='amazon'
)
print(f"Sentiment : {demo_en['sentiment']['sentiment']} {demo_en['sentiment']['emoji']} ({demo_en['sentiment']['confidence']})")
print(f"Topic     : {demo_en['topic']['primary_topic']}")
print(f"Phrases   : {[kp['phrase'] for kp in demo_en['keyphrases']['keyphrases'][:4]]}")
print(f"Summary   : {demo_en['summary']['summary']}")

print('\n--- Demo: Amharic Feedback ---')
demo_am = analyze_feedback(
    text='ይህ ምርት በጣም ጥሩ ነው። ፈጣን ማድረስ እና ጥሩ ጥራት አለው།',
    feedback_type='product_review',
    source='social_media'
)
print(f"Language  : {demo_am['input']['language']}")
print(f"Sentiment : {demo_am['sentiment']['sentiment']} {demo_am['sentiment']['emoji']}")
print(f"Topic     : {demo_am['topic']['primary_topic']}")

🖥️  Device: GPU
📥 Loading sentiment model...
📥 Loading zero-shot classifier...
📥 Loading summarizer...
📥 Loading KeyBERT...
✅ All models loaded!
✅ Full pipeline ready!

--- Demo: English Product Review ---
Sentiment : positive 😊 (0.9999)
Topic     : quality
Phrases   : ['delivery excellent', 'great quality', 'absolutely amazing', 'quality fast']
Summary   : This product is absolutely amazing! Great quality, fast delivery, and excellent customer support. Highly recommend!

--- Demo: Amharic Feedback ---
Language  : am
Sentiment : positive 😊
Topic     : quality


## 6. Flask REST API Server (Exposed via ngrok)

In [5]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading
import json

app = Flask(__name__)
CORS(app)

# ─── Health Check ─────────────────────────────────────────────────────────
@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'device': 'GPU' if device == 0 else 'CPU'})

# ─── Single Feedback Analysis ─────────────────────────────────────────────
@app.route('/api/analyze', methods=['POST'])
def api_analyze():
    """
    POST /api/analyze
    Body: { "text": "...", "feedback_type": "product_review", "source": "amazon" }
    """
    try:
        data = request.get_json(force=True)
        if not data or 'text' not in data:
            return jsonify({'error': 'Missing required field: text'}), 400

        result = analyze_feedback(
            text=data['text'],
            feedback_type=data.get('feedback_type', 'general'),
            source=data.get('source', 'unknown')
        )
        return jsonify(result), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# ─── Batch Analysis ───────────────────────────────────────────────────────
@app.route('/api/analyze/batch', methods=['POST'])
def api_batch_analyze():
    """
    POST /api/analyze/batch
    Body: { "feedbacks": [ {"text":"...", "feedback_type":"...", "source":"..."}, ... ] }
    """
    try:
        data = request.get_json(force=True)
        if not data or 'feedbacks' not in data:
            return jsonify({'error': 'Missing required field: feedbacks'}), 400

        feedbacks = data['feedbacks']
        if len(feedbacks) > 100:
            return jsonify({'error': 'Max batch size is 100'}), 400

        results  = batch_analyze(feedbacks)
        insights = generate_insights(results)
        return jsonify({'results': results, 'insights': insights, 'count': len(results)}), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# ─── Sentiment Only ───────────────────────────────────────────────────────
@app.route('/api/sentiment', methods=['POST'])
def api_sentiment():
    try:
        data = request.get_json(force=True)
        result = analyze_sentiment(data['text'])
        return jsonify(result), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# ─── Topics Only ──────────────────────────────────────────────────────────
@app.route('/api/topics', methods=['POST'])
def api_topics():
    try:
        data = request.get_json(force=True)
        result = classify_topic(data['text'], data.get('feedback_type', 'general'))
        return jsonify(result), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# ─── Key Phrases Only ─────────────────────────────────────────────────────
@app.route('/api/keyphrases', methods=['POST'])
def api_keyphrases():
    try:
        data = request.get_json(force=True)
        result = extract_keyphrases(data['text'], top_n=data.get('top_n', 8))
        return jsonify(result), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# ─── Summarize Only ───────────────────────────────────────────────────────
@app.route('/api/summarize', methods=['POST'])
def api_summarize():
    try:
        data = request.get_json(force=True)
        result = summarize_feedback(data['text'])
        return jsonify(result), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# ─── Insights (batch input) ───────────────────────────────────────────────
@app.route('/api/insights', methods=['POST'])
def api_insights():
    try:
        data = request.get_json(force=True)
        results  = batch_analyze(data['feedbacks'])
        insights = generate_insights(results)
        return jsonify(insights), 200
    except Exception as e:
        return jsonify({'error': str(e)}), 500

print('✅ Flask routes defined!')

✅ Flask routes defined!


## 7. Start Server & Expose via ngrok

In [6]:
import os
import time
import socket
import threading
import subprocess

# ── Install dependencies ───────────────────────────────────────────────────
subprocess.run(['pip', 'install', 'flask', 'pyngrok', '-q'], check=True)

from flask import Flask, request, jsonify
from pyngrok import ngrok

# ── Flask App + Routes ─────────────────────────────────────────────────────
app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'message': 'NLP Service is running'})

@app.route('/api/analyze', methods=['POST'])
def api_analyze():
    data = request.get_json()
    if not data or 'text' not in data:
        return jsonify({'error': 'Missing text field'}), 400
    result = analyze_feedback(
        text=data['text'],
        feedback_type=data.get('feedback_type', 'general'),
        source=data.get('source')
    )
    return jsonify(result)

@app.route('/api/analyze/batch', methods=['POST'])
def api_batch():
    data = request.get_json()
    if not data or 'feedbacks' not in data:
        return jsonify({'error': 'Missing feedbacks field'}), 400
    results = batch_analyze(data['feedbacks'])
    return jsonify({'results': results, 'count': len(results)})

@app.route('/api/sentiment', methods=['POST'])
def api_sentiment():
    data = request.get_json()
    if not data or 'text' not in data:
        return jsonify({'error': 'Missing text field'}), 400
    result = analyze_sentiment(data['text'])
    return jsonify(result)

@app.route('/api/topics', methods=['POST'])
def api_topics():
    data = request.get_json()
    if not data or 'text' not in data:
        return jsonify({'error': 'Missing text field'}), 400
    result = classify_topic(data['text'], data.get('feedback_type', 'general'))
    return jsonify(result)

@app.route('/api/keyphrases', methods=['POST'])
def api_keyphrases():
    data = request.get_json()
    if not data or 'text' not in data:
        return jsonify({'error': 'Missing text field'}), 400
    result = extract_keyphrases(data['text'])
    return jsonify(result)

@app.route('/api/summarize', methods=['POST'])
def api_summarize():
    data = request.get_json()
    if not data or 'text' not in data:
        return jsonify({'error': 'Missing text field'}), 400
    result = summarize_feedback(data['text'])
    return jsonify(result)

@app.route('/api/insights', methods=['POST'])
def api_insights():
    data = request.get_json()
    if not data or 'results' not in data:
        return jsonify({'error': 'Missing results field'}), 400
    result = generate_insights(data['results'])
    return jsonify(result)

# ── Kill port if already in use ────────────────────────────────────────────
os.system('fuser -k 5000/tcp')
time.sleep(1)

# ── Ngrok setup ────────────────────────────────────────────────────────────
NGROK_AUTH_TOKEN = '2repFCx5APaDv49xfYdw7JhY9a8_6cHiHvkjMycB38vqbLLyN'
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
time.sleep(1)

# ── Pick a free port ───────────────────────────────────────────────────────
def find_free_port(preferred=5000):
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.bind(('', preferred))
            return preferred
    except OSError:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.bind(('', 0))
            return s.getsockname()[1]

PORT = find_free_port(5000)
print(f'🔌 Using port: {PORT}')

# ── Start Flask in background thread ──────────────────────────────────────
def run_flask():
    app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

# ── Open ngrok tunnel ──────────────────────────────────────────────────────
public_url = ngrok.connect(PORT).public_url
print(f'\n🚀 NLP Server is LIVE!')
print(f'📡 Public URL : {public_url}')
print(f'\n📋 Copy this URL to your Spring Boot application.properties:')
print(f'   nlp.service.base-url={public_url}')
print(f'\n🔗 API Endpoints:')
print(f'   POST {public_url}/api/analyze')
print(f'   POST {public_url}/api/analyze/batch')
print(f'   POST {public_url}/api/sentiment')
print(f'   POST {public_url}/api/topics')
print(f'   POST {public_url}/api/keyphrases')
print(f'   POST {public_url}/api/summarize')
print(f'   POST {public_url}/api/insights')
print(f'   GET  {public_url}/health')

🔌 Using port: 5000
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



🚀 NLP Server is LIVE!
📡 Public URL : https://f882-35-252-172-85.ngrok-free.app

📋 Copy this URL to your Spring Boot application.properties:
   nlp.service.base-url=https://f882-35-252-172-85.ngrok-free.app

🔗 API Endpoints:
   POST https://f882-35-252-172-85.ngrok-free.app/api/analyze
   POST https://f882-35-252-172-85.ngrok-free.app/api/analyze/batch
   POST https://f882-35-252-172-85.ngrok-free.app/api/sentiment
   POST https://f882-35-252-172-85.ngrok-free.app/api/topics
   POST https://f882-35-252-172-85.ngrok-free.app/api/keyphrases
   POST https://f882-35-252-172-85.ngrok-free.app/api/summarize
   POST https://f882-35-252-172-85.ngrok-free.app/api/insights
   GET  https://f882-35-252-172-85.ngrok-free.app/health


## 8. Sample Test Calls

In [9]:
import requests as req

BASE = 'http://localhost:5000'  # change to ngrok public_url if needed


def safe_post(url, payload):
    r = req.post(url, json=payload)

    # 🔴 Prevent JSONDecodeError
    if r.status_code != 200:
        print("\n❌ Request Failed")
        print("Status Code:", r.status_code)
        print("Response:", r.text)
        return None

    try:
        return r.json()
    except Exception as e:
        print("\n❌ Invalid JSON Response:", e)
        print("Raw Response:", r.text)
        return None


# ── Test 1: Full analysis ─────────────────────────────
result = safe_post(f'{BASE}/api/analyze', {
    'text': 'The delivery was delayed by 2 weeks and the package arrived damaged. Very disappointed.',
    'feedback_type': 'product_review',
    'source': 'website'
})

if result:
    print('=== Full Analysis Test ===')
    print(f"Sentiment  : {result['sentiment']['sentiment']} {result['sentiment']['emoji']}")
    print(f"Topic      : {result['topic']['primary_topic']}")
    print(f"Key Phrases: {[k.get('phrase') for k in result['keyphrases'].get('keyphrases', [])[:3]]}")
    print(f"Summary    : {result['summary']['summary']}")


# ── Test 2: Amharic support ticket ───────────────────
r2_data = safe_post(f'{BASE}/api/analyze', {
    'text': 'ለምን ደሞዜን አትከፍሉኝም?',
    'feedback_type': 'support_ticket',
    'source': 'email'
})

if r2_data:
    print('\n=== Amharic Support Ticket Test ===')
    print(f"Language   : {r2_data.get('input', {}).get('language', 'unknown')}")
    print(f"Sentiment  : {r2_data['sentiment']['sentiment']} {r2_data['sentiment']['emoji']}")
    print(f"Topic      : {r2_data['topic']['primary_topic']}")

INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 13:22:42] "POST /api/analyze HTTP/1.1" 200 -


=== Full Analysis Test ===
Sentiment  : negative 😞
Topic      : delivery
Key Phrases: ['delivery delayed', 'arrived damaged', 'damaged disappointed']
Summary    : The delivery was delayed by 2 weeks and the package arrived damaged. Very disappointed.


INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 13:22:43] "POST /api/analyze HTTP/1.1" 200 -



=== Amharic Support Ticket Test ===
Language   : am
Sentiment  : negative 😞
Topic      : billing issue
